# Test Prediction Request ke Model Serving
### Marketing Campaign Response Prediction — TF Serving

Notebook ini menguji model yang sudah di-serve oleh **TensorFlow Serving**
(lihat `Dockerfile`, dijalankan via `docker compose up --build` atau setelah
di-deploy ke Heroku — lihat `DEPLOYMENT.md`).

Model menerima input dalam format `tf.Example` yang di-serialize sebagai
base64 string di dalam JSON request (format standar REST API TF Serving
untuk signature yang menerima `serialized_tf_examples`).

**Sebelum menjalankan notebook ini**, pastikan container TF Serving sudah
jalan, misalnya:
```bash
PORT=8501 docker compose up --build
```
atau kalau menguji app yang sudah di-deploy ke Heroku, ganti `BASE_URL` di
bawah dengan URL Heroku app kamu.

In [ ]:
import requests
import json
import base64
import tensorflow as tf
import pandas as pd

# Ganti dengan URL Heroku app kamu kalau menguji yang sudah di-deploy ke cloud,
# contoh: BASE_URL = "https://harmanm-marketing-response.herokuapp.com"
BASE_URL = "http://localhost:8501"
MODEL_NAME = "marketing-response-model"


## 1. Cek Status Model
Langkah pertama: pastikan model benar-benar termuat dan siap (`state: AVAILABLE`) sebelum mengirim request prediksi.

In [ ]:
status_url = f"{BASE_URL}/v1/models/{MODEL_NAME}"
response = requests.get(status_url)
print("Status code:", response.status_code)
print(json.dumps(response.json(), indent=2))

## 2. Siapkan Data Contoh untuk Prediksi
Ambil beberapa baris dari data yang sudah dibersihkan (`data/marketing_campaign_clean.csv`)
sebagai contoh pelanggan yang akan diprediksi. Kolom `Response` (label asli)
disimpan terpisah hanya untuk pembanding — TIDAK dikirim ke model.

In [ ]:
df = pd.read_csv('data/marketing_campaign_clean.csv')

# Ambil 3 baris contoh: 1 yang tahu labelnya Response=1, 1 yang Response=0, 1 acak
sample_df = pd.concat([
    df[df['Response'] == 1].head(1),
    df[df['Response'] == 0].head(1),
    df.sample(1, random_state=42),
]).reset_index(drop=True)

actual_labels = sample_df['Response'].tolist()
sample_df_for_request = sample_df.drop(columns=['Response'])
sample_df_for_request

## 3. Konversi ke Format `tf.Example` (Base64)
TF Serving REST API mengharapkan input dalam bentuk JSON dengan field
`examples` berisi list string ter-`base64` dari `tf.Example` yang sudah
di-serialize — ini karena signature model (`serve_tf_examples_fn` di
`marketing_trainer.py`) menerima input `serialized_tf_examples`.

In [ ]:
def df_row_to_tf_example(row: pd.Series) -> tf.train.Example:
    feature = {}
    for col, val in row.items():
        if pd.api.types.is_float_dtype(type(val)) or isinstance(val, float):
            feature[col] = tf.train.Feature(float_list=tf.train.FloatList(value=[float(val)]))
        elif isinstance(val, (int,)):
            feature[col] = tf.train.Feature(int64_list=tf.train.Int64List(value=[int(val)]))
        else:
            feature[col] = tf.train.Feature(bytes_list=tf.train.BytesList(value=[str(val).encode('utf-8')]))
    return tf.train.Example(features=tf.train.Features(feature=feature))


def build_request_payload(dataframe: pd.DataFrame) -> dict:
    examples_b64 = []
    for _, row in dataframe.iterrows():
        tf_example = df_row_to_tf_example(row)
        serialized = tf_example.SerializeToString()
        examples_b64.append({"b64": base64.b64encode(serialized).decode('utf-8')})
    return {"instances": examples_b64}


payload = build_request_payload(sample_df_for_request)
print("Jumlah instance yang dikirim:", len(payload['instances']))

## 4. Kirim Prediction Request
Endpoint REST API standar TF Serving: `POST /v1/models/<model_name>:predict`.

In [ ]:
predict_url = f"{BASE_URL}/v1/models/{MODEL_NAME}:predict"
response = requests.post(predict_url, data=json.dumps(payload))

print("Status code:", response.status_code)
result = response.json()
print(json.dumps(result, indent=2))

## 5. Interpretasi Hasil
Output model adalah probabilitas (`sigmoid`) bahwa pelanggan akan merespons
campaign (`Response = 1`). Bandingkan dengan label asli untuk sanity check.

In [ ]:
predictions = result['predictions']

for i, (pred, actual) in enumerate(zip(predictions, actual_labels)):
    prob = pred[0] if isinstance(pred, list) else pred
    predicted_label = 1 if prob >= 0.5 else 0
    match = "✅ cocok" if predicted_label == actual else "❌ beda"
    print(f"Sample {i+1}: probabilitas Response=1 -> {prob:.4f} | prediksi: {predicted_label} | label asli: {actual} | {match}")

## 6. (Opsional) Test dengan Data Custom
Contoh mengirim satu data pelanggan buatan sendiri, bukan dari dataset,
untuk simulasi pemakaian nyata (misalnya dari form input aplikasi).

In [ ]:
custom_customer = pd.DataFrame([{
    'Education': 'Graduation',
    'Marital_Status': 'Married',
    'Income': 65000.0,
    'Kidhome': 0,
    'Teenhome': 0,
    'Recency': 5,
    'MntWines': 500,
    'MntFruits': 50,
    'MntMeatProducts': 300,
    'MntFishProducts': 80,
    'MntSweetProducts': 40,
    'MntGoldProds': 60,
    'NumDealsPurchases': 2,
    'NumWebPurchases': 6,
    'NumCatalogPurchases': 4,
    'NumStorePurchases': 8,
    'NumWebVisitsMonth': 3,
    'AcceptedCmp3': 0, 'AcceptedCmp4': 1, 'AcceptedCmp5': 0, 'AcceptedCmp1': 0, 'AcceptedCmp2': 0,
    'Complain': 0,
    'Age': 42,
    'Customer_Tenure_Days': 400,
    'Total_Spending': 1030,
    'Total_Children': 0,
    'Total_Purchases': 20,
}])

custom_payload = build_request_payload(custom_customer)
response = requests.post(predict_url, data=json.dumps(custom_payload))
result = response.json()
prob = result['predictions'][0][0] if isinstance(result['predictions'][0], list) else result['predictions'][0]
print(f"Probabilitas pelanggan ini merespons campaign: {prob:.4f}")
print("Prediksi:", "Kemungkinan merespons (1)" if prob >= 0.5 else "Kemungkinan tidak merespons (0)")

## Kesimpulan

Model yang di-serve lewat TensorFlow Serving berhasil diuji dengan mengirim
prediction request langsung ke REST API-nya (`/v1/models/{model}:predict`),
membuktikan bahwa:
1. Model dapat menerima request eksternal dalam format standar (`tf.Example` ter-base64).
2. Output probabilitas konsisten dengan hasil evaluasi model di notebook pipeline utama.
3. Sistem end-to-end (pipeline TFX → model serving → prediction request) berjalan utuh.